# Brain Metastasis Segmentation - 16³ Patch Training

**Setup:** Run cells 1-5 in order, then run cell 6 to train.

**Timeline:**
- Cells 1-3: ~2 min (setup)
- Cell 4: ~10-15 min (copy data to local SSD)
- Cell 5: ~1 min (setup)
- Cell 6: ~25 min pre-load + ~10 min training = ~35 min total

**If session disconnects:** Run cells 1-5, then cell 6b to resume.

In [ ]:
# 1. Check GPU & Mount Drive
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/brainMetShare'
DRIVE_DIR = '/content/drive/MyDrive/BrainMetShare'
LOCAL_DATA = '/content/data'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model", exist_ok=True)

In [ ]:
# 2. Install Dependencies
!pip install nibabel scipy tqdm tensorboard pyyaml monai -q
print("Dependencies installed!")

In [ ]:
# 3. Upload Code
from google.colab import files
print("Upload brainMetShare_code.zip")
uploaded = files.upload()
!unzip -o brainMetShare_code.zip -d /content/brainMetShare
print("Code extracted!")

In [ ]:
# 4. Copy Data to Local SSD (Drive is too slow)
import time
start = time.time()

!df -h /content

print("\nCopying train data...")
!cp -r {DRIVE_DIR}/preprocessed_256/train {LOCAL_DATA}/train
print(f"Train: {len(os.listdir(LOCAL_DATA + '/train'))} cases")

print("Copying test data...")
!cp -r {DRIVE_DIR}/preprocessed_256/test {LOCAL_DATA}/test
print(f"Test: {len(os.listdir(LOCAL_DATA + '/test'))} cases")

print(f"\nData copied in {(time.time()-start)/60:.1f} min")
!df -h /content

In [ ]:
# 5. Setup Python Path
import sys
sys.path.insert(0, '/content/brainMetShare/src')
os.chdir(PROJECT_DIR)
print("Ready for training!")

In [ ]:
# 6. Train (Pre-load + Fast Training)
from segmentation.dataset import BrainMetDataset
from segmentation.unet import LightweightUNet3D
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch
import torch.nn as nn
from tqdm import tqdm
import time
import json
from datetime import datetime

# Config
BATCH_SIZE = 64
EPOCHS = 150
LR = 0.001
PATCH_SIZE = (16, 16, 16)

# Load dataset
ds = BrainMetDataset(
    data_dir='/content/data/train',
    sequences=['t1_pre', 't1_gd', 'flair', 't2'],
    patch_size=PATCH_SIZE,
    target_size=None,
    transform=None
)

# Pre-load ALL samples into RAM
print(f"\nPre-loading {len(ds)} samples into RAM...")
start = time.time()
all_images = []
all_masks = []
for i in tqdm(range(len(ds))):
    img, mask, _ = ds[i]
    all_images.append(img)
    all_masks.append(mask)

images_tensor = torch.stack(all_images)
masks_tensor = torch.stack(all_masks)
del all_images, all_masks  # Free memory
print(f"Pre-loaded in {(time.time()-start)/60:.1f} min")
print(f"Images: {images_tensor.shape} ({images_tensor.element_size() * images_tensor.numel() / 1e6:.1f} MB)")

# Train/val split (85/15)
cached_ds = TensorDataset(images_tensor, masks_tensor)
train_size = int(0.85 * len(cached_ds))
val_size = len(cached_ds) - train_size
train_ds, val_ds = random_split(cached_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Batches/epoch: {len(train_loader)}")

# Model
model = LightweightUNet3D(in_channels=4, out_channels=1).cuda()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer, scheduler, loss
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()

# Training
best_val_loss = float('inf')
model_path = f"{PROJECT_DIR}/model/tiny_lesion_16patch_best.pth"
state_path = f"{PROJECT_DIR}/model/tiny_lesion_16patch_state.json"

print(f"\n{'='*50}")
print(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LR}")
print(f"{'='*50}\n")

train_start = time.time()
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for img, mask in train_loader:
        img, mask = img.cuda(), mask.cuda()
        optimizer.zero_grad()
        out = model(img)
        loss = criterion(out, mask)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    # Validate
    model.eval()
    val_loss = 0
    val_dice = 0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.cuda(), mask.cuda()
            out = model(img)
            val_loss += criterion(out, mask).item()
            # Dice
            pred = (torch.sigmoid(out) > 0.5).float()
            intersection = (pred * mask).sum()
            dice = (2 * intersection + 1e-6) / (pred.sum() + mask.sum() + 1e-6)
            val_dice += dice.item()
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    
    scheduler.step()
    
    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_dice': val_dice
        }, model_path)
    
    # Save state for monitoring
    with open(state_path, 'w') as f:
        json.dump({
            'epoch': epoch,
            'epochs_total': EPOCHS,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_dice': val_dice,
            'best_val_loss': best_val_loss,
            'timestamp': datetime.now().isoformat()
        }, f, indent=2)
    
    epoch_time = time.time() - epoch_start
    if epoch % 10 == 0 or epoch <= 3:
        print(f"Epoch {epoch:3d}/{EPOCHS}: train={train_loss:.4f}, val={val_loss:.4f}, dice={val_dice:.3f}, best={best_val_loss:.4f} ({epoch_time:.1f}s)")

total_time = (time.time() - train_start) / 60
print(f"\n{'='*50}")
print(f"Training complete in {total_time:.1f} min")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Model saved: {model_path}")
print(f"{'='*50}")

In [ ]:
# 6b. Resume Training (if session disconnected)
# Run cells 1-5 first, then run this cell

# from segmentation.dataset import BrainMetDataset
# from segmentation.unet import LightweightUNet3D
# ... (copy the training code from cell 6, add checkpoint loading)

In [ ]:
# 7. Check Progress
import json
state_file = f"{PROJECT_DIR}/model/tiny_lesion_16patch_state.json"
if os.path.exists(state_file):
    with open(state_file, 'r') as f:
        state = json.load(f)
    print(f"Epoch: {state['epoch']}/{state['epochs_total']}")
    print(f"Train Loss: {state['train_loss']:.4f}")
    print(f"Val Loss: {state['val_loss']:.4f}")
    print(f"Val Dice: {state['val_dice']:.4f}")
    print(f"Best Val Loss: {state['best_val_loss']:.4f}")
else:
    print("No training state yet.")

In [ ]:
# 8. Save Model to Drive
import shutil
from datetime import datetime

save_dir = f"{DRIVE_DIR}/models/tiny_lesion_16patch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(save_dir, exist_ok=True)

for f in ['tiny_lesion_16patch_best.pth', 'tiny_lesion_16patch_state.json']:
    src = f"{PROJECT_DIR}/model/{f}"
    if os.path.exists(src):
        shutil.copy(src, save_dir)
        print(f"Saved: {f}")

print(f"\nModel saved to: {save_dir}")